# ECMWF AIFS-single — Download

**Model:** ECMWF Artificial Intelligence/Integrated Forecasting System (AIFS) — single deterministic run  
**Type:** AI/ML-based deterministic  
**Cycles:** 00Z and 12Z  
**Range:** 0–360h

## Parameter differences vs IFS HRES

| IFS param | AIFS equivalent | Notes |
|-----------|-----------------|-------|
| `tcwv` | `tcw` | Same quantity, different GRIB name |
| `10fg` | — | Not available |
| `ro`, `sp`, `mucape` | — | Not available |
| `r` (RH), `vo`, `d` | — | Not in upper-air output |

## Files produced

```
data/YYYY-MM-DD_HHZ/
    sfc_f000-f120_3h.grib2   surface (3-hourly, 0–120h)
    pl_f000-f120_12h.grib2   upper air (12-hourly, 0–120h)
```

In [ ]:
from ecmwf.opendata import Client
from pathlib import Path

BASE_DIR = Path("data")
BASE_DIR.mkdir(parents=True, exist_ok=True)
print("Ready. BASE_DIR:", BASE_DIR.resolve())

In [ ]:
client = Client(source="ecmwf", model="aifs-single")
latest_run  = client.latest()
latest_time = latest_run.hour

run_label  = f"{latest_run.strftime('%Y-%m-%d')}_{latest_time:02d}Z"
output_dir = BASE_DIR / run_label
output_dir.mkdir(parents=True, exist_ok=True)

print("Latest run  :", latest_run)
print("Cycle (UTC) :", latest_time)
print("Output dir  :", output_dir.resolve())

In [ ]:
# ── Surface fields — 3-hourly, 0–120h ────────────────────────────────────────
steps_3h = list(range(0, 121, 3))

SFC = {}
SFC["thermo"]   = ["2t", "2d", "skt", "sst"]
SFC["wind"]     = ["10u", "10v", "100u", "100v"]
SFC["pressure"] = ["msl"]
SFC["moisture"] = ["tp", "cp", "tcc", "lcc", "mcc", "hcc", "tcw"]
SFC["energy"]   = ["ssrd", "strd"]
# Note: AIFS-single has no blh, ttr, mucape, mucin, 10fg, sp, ro

client.retrieve(
    time=latest_time, step=steps_3h, type="fc",
    param=[p for grp in SFC.values() for p in grp],
    target=str(output_dir / "sfc_f000-f120_3h.grib2"),
)
print("Saved: sfc_f000-f120_3h.grib2")
print(f"  Groups: { {k: len(v) for k, v in SFC.items()} }")

In [ ]:
# ── Upper-air — 12-hourly, 0–120h ────────────────────────────────────────────
# AIFS does not output r (relative humidity), vo (vorticity), or d (divergence)
PL = {}
PL["thermo"]   = ["t", "gh"]
PL["wind"]     = ["u", "v", "w"]
PL["moisture"] = ["q"]

client.retrieve(
    time=latest_time, step=list(range(0, 121, 12)), type="fc",
    param=[p for grp in PL.values() for p in grp],
    levtype="pl", levelist=[1000, 925, 850, 700, 600, 500, 400, 300, 250, 200],
    target=str(output_dir / "pl_f000-f120_12h.grib2"),
)
print("Saved: pl_f000-f120_12h.grib2")
print(f"  Groups: { {k: len(v) for k, v in PL.items()} }")